# NB_hospital_daily_summary

Builds `adb_rapollocare.gold.hospital_daily_summary` (one row per hospital per day with lab activity) from `silver.lab_results` per the Silver-to-Gold STTM, LEFT JOINed to hospitals and calendar, with a LAG-based day-over-day trend. Load type: FULL (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
%sql
CREATE OR REPLACE TABLE adb_rapollocare.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab activity, incl. day-over-day trend. Built by NB_hospital_daily_summary (FULL rebuild).'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                               AS summary_date,
    count(*)                                AS total_tests,
    count(DISTINCT patient_id)              AS unique_patients_tested,
    count(DISTINCT doctor_id)               AS active_doctor_count,
    count_if(result_status = 'Abnormal')    AS abnormal_test_count,
    count_if(result_status = 'Critical')    AS critical_test_count
  FROM adb_rapollocare.silver.lab_results
  GROUP BY hospital_id, test_date
),
trend AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  t.hospital_id,
  h.hospital_name,
  t.summary_date,
  c.day_name,
  c.is_weekend,
  CAST(t.total_tests AS INT)                                              AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                   AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                      AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                      AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                      AS critical_test_count,
  CAST(round(t.abnormal_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(round(t.critical_test_count * 100.0 / t.total_tests, 2) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                     AS prev_day_total_tests,
  CASE
    WHEN t.prev_day_total_tests IS NULL OR t.prev_day_total_tests = 0 THEN NULL
    ELSE CAST(round((t.total_tests - t.prev_day_total_tests) * 100.0 / t.prev_day_total_tests, 2) AS DECIMAL(6,2))
  END                                                                     AS test_volume_change_pct
FROM trend t
LEFT JOIN adb_rapollocare.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN adb_rapollocare.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT);

In [0]:
# Return the gold table record count to the caller (job / dbutils.notebook.run)
updated_df = spark.table("adb_rapollocare.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))